In [ ]:
from spiral import Spiral
import pyarrow as pa

sp = Spiral()
project = sp.project("external-805943")

# Random IDs are discouraged; use monotonically increasing IDs where possible, like UUIDv7.
table = project.create_table("experiments-v0", key_schema=pa.schema({"data_key": pa.string()}), exist_ok=True)

In [38]:
import numpy as np
from pathlib import Path
import yaml
import json

# Loads experiment data from a directory. Download experiment referenced in `inspect_data.ipynb`.
experiment_data_path = Path("35_3832003544063_1_V1")

with open(experiment_data_path / "meta.json", "r") as f:
    global_meta = json.load(f)

In [43]:
eye_tracker_means = np.load(experiment_data_path / "eye_tracker" / "meta" / "means.npy")
eye_tracker_stds = np.load(experiment_data_path / "eye_tracker" / "meta" / "stds.npy")
with open(experiment_data_path / "eye_tracker" / "meta.yml", "r") as f:
    eye_tracker_meta = yaml.safe_load(f)
print(eye_tracker_stds)

[  83.20033612   66.84043318 2292.67757777]


In [47]:
responses_30hz_no_filtering_means = np.load(experiment_data_path / "responses_30Hz_no_filtering" / "meta" / "means.npy")
responses_30hz_no_filtering_stds = np.load(experiment_data_path / "responses_30Hz_no_filtering" / "meta" / "stds.npy")
with open(experiment_data_path / "responses_30Hz_no_filtering" / "meta.yml", "r") as f:
    responses_30hz_no_filtering_meta = yaml.safe_load(f)

0.05422926152272893


In [ ]:
responses_30Hz_permissive_stability_criteria_means = np.load(experiment_data_path / "responses_30Hz_permissive_stability_criteria" / "meta" / "means.npy")
responses_30Hz_permissive_stability_criteria_stds = np.load(experiment_data_path / "responses_30Hz_permissive_stability_criteria" / "meta" / "stds.npy")
with open(experiment_data_path / "responses_30Hz_permissive_stability_criteria" / "meta.yml", "r") as f:
    responses_30Hz_permissive_stability_criteria_meta = yaml.safe_load(f)

In [ ]:
screen_timestamps = np.load(experiment_data_path / "screen" / "timestamps.npy")
with open(experiment_data_path / "screen" / "meta.yml", "r") as f:
    screen_meta = yaml.safe_load(f)

# FIXME(marko): Figure out cell sizing.
#   ArrowCapacityError: array cannot contain more than 2147483646 bytes, have 2152886400
screen_limit = 50

screen_data_meta = []
for i, frame_meta_path in enumerate(sorted((experiment_data_path / "screen" / "meta").glob("*.yml"))):
    if i == screen_limit:
        break
    with open(frame_meta_path, "r") as f:
        screen_data_meta.append(yaml.safe_load(f))

print(f"Loaded metadata for {len(screen_data_meta)} frames")

screen_data = []
for i, frame_path in enumerate(sorted((experiment_data_path / "screen" / "data").glob("*.npy"))):
    if i == screen_limit:
        break
    # Store frames as bytes
    screen_data.append(np.load(frame_path).tobytes(order='C'))

print(f"Loaded data for {len(screen_data)} frames")

In [ ]:
experiment = dict(global_meta)
experiment["eye_tracker"] = dict(eye_tracker_meta)
experiment["eye_tracker"]["data"] = {
    "means": eye_tracker_means,
    "stds": eye_tracker_stds,
}
experiment["responses_30Hz_no_filtering"] = dict(responses_30hz_no_filtering_meta)
experiment["responses_30Hz_no_filtering"]["data"] = {
    "means": responses_30hz_no_filtering_means,
    "stds": responses_30hz_no_filtering_stds,
}
experiment["responses_30Hz_permissive_stability_criteria"] = dict(responses_30Hz_permissive_stability_criteria_meta)
experiment["responses_30Hz_permissive_stability_criteria"]["data"] = {
    "means": responses_30Hz_permissive_stability_criteria_means,
    "stds": responses_30Hz_permissive_stability_criteria_stds,
}
experiment["screen"] = dict(screen_meta)
experiment["screen"]["timestamps"] = screen_timestamps
experiment["screen"]["metas"] = screen_data_meta
experiment["screen"]["data"] = {
    "frames": screen_data
}

In [ ]:
# StructType(
#   data_key
#
#   config:
#     database: {host, password, schema_name, user}
#     export: {compute_report, default_fs, export_suffix, output_base_dir, overwrite,
#              quality_thresholds: {good_spikes_fraction, loo_slope, presence_ratio, variance_explained},
#              target_sampling_rate, use_functional_criteria, use_regularity_criteria, use_stability_criteria}
#     gaze: {behavior_mnt_dir: list[str], blink_acc_threshold, blink_detection}
#     processing: {n_jobs}
#     screen: {dtype, frame_t_max, frame_t_min, frames_in_split, gray_value, max_delta_pauses, new_h, new_w,
#              normalization: {means: list[double], stds: list[double]},
#              original_video_h, post_stim_gray_s, pre_stim_gray_s, save_as_normalized}
#     session: {auto_discover, beh_offset, beh_path, brain_area,
#               database_restrictions: {correct_trials},
#               electrode_id, export_gaze, export_screen, name, session_start_time,
#               spike_path, stim_path, subject_id,
#               subject_to_animal: {35, 37},
#               test_tier: null, timestamp_tolerance_seconds, train_tier: null}
#     stability: {enforce_refractory_period, presence_ratio_bin_size_s}
#
#   eye_tracker:
#     data: {means: list[double], stds: list[double]}
#     {dtype, end_time, is_mem_mapped, modality, n_signals, n_timestamps, phase_shift_per_signal, sampling_rate, start_time}
#
#   responses_30Hz_no_filtering:
#     data: {means: list[double], stds: list[double]}
#     {dtype: null, end_time, is_mem_mapped, modality, n_signals, n_timestamps, phase_shift_per_signal, sampling_rate, start_time}
#
#   responses_30Hz_permissive_stability_criteria:
#     data: {means: list[double], stds: list[double]}
#     {dtype: null, end_time, is_mem_mapped, modality, n_signals, n_timestamps, phase_shift_per_signal, sampling_rate, start_time}
#
#   screen:
#     data: {frames: list[binary]}
#     metas: list[{first_frame_idx, image_size: list[int64], interleave_value, modality, movie_id, num_frames, tier, trial_idx, valid_trial}]
#     {modality, timestamps: list[double]}
# )

In [ ]:
table.write([experiment])

In [ ]:
table.schema()